# 🧪 Nghiên cứu Mô hình Nâng cao — Kernel Ridge Regression (KRR)

**Dự án:** TourismFit-OLS — Hồi quy và Dự báo chi phí du lịch  
**Bộ dữ liệu:** Tanzania Tourism Expenditure (Zindi)  
**Biến mục tiêu:** `total_cost` (Tổng chi phí du lịch thực tế, đơn vị TZS)  

---

## Mục tiêu nghiên cứu
1. Thực hiện **Phân tích Khám phá Dữ liệu (EDA)** trực quan để xác nhận đặc tính phân phối của biến mục tiêu và các đặc trưng số học.
2. Xây dựng mô hình phi tuyến **Kernel Ridge Regression (KRR)** tích hợp hạt nhân RBF (Radial Basis Function) để mô hình hóa các mối quan hệ phi tuyến phức tạp trong dữ liệu chi tiêu du lịch.
3. Thiết lập giải pháp số học hiệu năng cao cho hệ phương trình tuyến tính đối ngẫu bằng phương pháp **khử Gauss chọn phần tử trội từng phần (Gaussian Elimination with Partial Pivoting)** tự cài đặt.
4. Cài đặt thuật toán **Lựa chọn đặc trưng OLS + Feature Selection** loại bỏ tuần tự đa cộng tuyến dựa trên ngưỡng VIF để cải thiện tính vững của mô hình tuyến tính.
5. Áp dụng quy trình kiểm định chéo **5-Fold Cross-Validation** kết hợp **Grid Search** để tối ưu hóa siêu tham số cho cả Ridge, Lasso và Kernel Ridge Regression.
6. Đối chiếu hiệu năng giữa các mô hình (**OLS, OLS + VIF, Ridge, Lasso, KRR**) dựa trên các chỉ số **MAE, RMSE, và $R^2$ trên thang đo gốc (TZS)**, vẽ biểu đồ chẩn đoán phần dư và độ quan trọng đặc trưng.


## Phase 1: Khai báo thư viện và Tích hợp Pipeline dữ liệu
Nạp các thư viện phục vụ tính toán số học, quản lý dữ liệu, trực quan hóa đồ họa và các module tiền xử lý dữ liệu của hệ thống.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from typing import Tuple, List, Dict, Optional

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Thiết lập thư mục làm việc để import các file trong src/
notebook_dir = os.getcwd()
src_dir = os.path.abspath(os.path.join(notebook_dir, "src"))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from data_pipeline import DataPipeline, PipelineConfig
print("✓ Import DataPipeline thành công!")


## Phase 2: Nạp và Tiền xử lý dữ liệu
Khởi tạo cấu hình `PipelineConfig` và thực hiện tiền xử lý toàn diện. Dữ liệu được điền khuyết (Mean/Mode), mã hóa One-Hot, chuẩn hóa Z-score các đặc trưng liên tục và chèn cột hằng số (`intercept`) vào vị trí đầu tiên.


In [ ]:
config = PipelineConfig(data_dir="data", missing_method="mean")
pipeline = DataPipeline(config)
pipe_result = pipeline.run()

X_train = pipe_result.X_train
y_train = pipe_result.y_train
X_test = pipe_result.X_test
feature_names = pipe_result.feature_names

# Xử lý các giá trị khuyết còn sót lại để mô hình không gặp lỗi số học
X_train = np.nan_to_num(X_train, nan=0.0)
X_test = np.nan_to_num(X_test, nan=0.0)

print(f"\nDữ liệu đã sẵn sàng:")
print(f"  - X_train: {X_train.shape} (bao gồm cả intercept)")
print(f"  - y_train: {y_train.shape}")
print(f"  - X_test:  {X_test.shape}")
print(f"  - Số đặc trưng: {len(feature_names)}")


### Phase 2.1: Phân tích Khám phá Dữ liệu trực quan (EDA)
Khảo sát các thống kê mô tả, kiểm tra hiện tượng lệch (skewness) của biến mục tiêu `total_cost` và tính toán ma trận tương quan giữa các biến liên tục.


In [ ]:
# 1. Thống kê mô tả các đặc trưng liên tục (loại trừ intercept ở cột 0)
num_cols = ["total_female", "total_male", "night_mainland", "night_zanzibar"]
df_num = pd.DataFrame(X_train[:, 1:5], columns=num_cols)
print("=" * 60)
print("THỐNG KÊ MÔ TẢ CÁC ĐẶC TRƯNG SỐ HỌC (Z-SCORE)")
print("=" * 60)
print(df_num.describe())

# 2. Trực quan hóa phân phối biến mục tiêu (total_cost) - Gốc vs Log1p
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("PHÂN TÍCH PHÂN PHỐI BIẾN MỤC TIÊU TOTAL_COST", fontsize=14, fontweight="bold", y=1.05)

# Phân phối gốc
ax = axes[0]
ax.hist(y_train, bins=50, color="teal", edgecolor="black", alpha=0.75)
ax.set_title(f"Phân phối gốc (Original Scale)\nSkewness: {pd.Series(y_train).skew():.2f}", fontweight="bold")
ax.set_xlabel("total_cost (TZS)")
ax.set_ylabel("Tần suất")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

# Phân phối Log1p
ax = axes[1]
y_train_log = np.log1p(y_train)
ax.hist(y_train_log, bins=50, color="teal", edgecolor="black", alpha=0.75)
ax.set_title(f"Phân phối sau Log1p\nSkewness: {pd.Series(y_train_log).skew():.2f}", fontweight="bold")
ax.set_xlabel("log1p(total_cost)")
ax.set_ylabel("Tần suất")

# Boxplot gốc tìm Outliers
ax = axes[2]
ax.boxplot(y_train, patch_artist=True,
           boxprops=dict(facecolor="teal", color="black", alpha=0.6),
           medianprops=dict(color="red", linewidth=2),
           flierprops=dict(marker="o", markersize=4, alpha=0.3))
ax.set_title("Boxplot Tìm Outliers", fontweight="bold")
ax.set_ylabel("total_cost (TZS)")
ax.set_xticklabels(["total_cost"])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

plt.tight_layout()
plt.show()

# 3. Heatmap ma trận tương quan biến số liên tục
plt.figure(figsize=(8, 6))
sns.heatmap(df_num.corr(), annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Ma Trận Tương Quan Hệ Số Pearson", fontsize=12, fontweight="bold", pad=12)
plt.show()


## Phase 3: Cơ sở Toán học của Kernel Ridge Regression
Hồi quy tuyến tính Ridge tối thiểu hóa tổng sai số bình phương kèm theo số hạng phạt chính quy hóa $L_2$:
$$\mathcal{L}(\boldsymbol{\beta}) = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|_2^2 + \alpha \|\boldsymbol{\beta}\|_2^2$$

Theo Định lý Đại diện (Representer Theorem), nghiệm tối ưu $\boldsymbol{\beta}$ được biểu diễn dưới dạng tổ hợp tuyến tính của các hàng trong ma trận thiết kế $\mathbf{X}$:
$$\boldsymbol{\beta} = \mathbf{X}^T \mathbf{c}$$
trong đó $\mathbf{c} \in \mathbb{R}^n$ là vector hệ số đối ngẫu (dual coefficients).

Áp dụng hàm hạt nhân phi tuyến $k(\mathbf{x}_i, \mathbf{x}_j) = \Phi(\mathbf{x}_i)^T \Phi(\mathbf{x}_j)$, ma trận hạt nhân Gram $\mathbf{K} \in \mathbb{R}^{n \times n}$ được xác định với thành phần $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$. Hàm mục tiêu đối ngẫu cần tối thiểu hóa theo $\mathbf{c}$ chuyển thành:
$$\mathcal{L}(\mathbf{c}) = \mathbf{c}^T \mathbf{K}^2 \mathbf{c} - 2 \mathbf{c}^T \mathbf{K} \mathbf{y} + \mathbf{y}^T \mathbf{y} + \alpha \mathbf{c}^T \mathbf{K} \mathbf{c}$$

Đạo hàm theo vector hệ số đối ngẫu $\mathbf{c}$ và cho bằng 0 dẫn đến hệ phương trình tuyến tính đối ngẫu:
$$(\mathbf{K} + \alpha \mathbf{I}_n) \mathbf{c} = \mathbf{y}$$

Nghiệm hệ số đối ngẫu được tìm bằng cách giải hệ phương trình:
$$\mathbf{c} = (\mathbf{K} + \alpha \mathbf{I}_n)^{-1} \mathbf{y}$$

Khi thực hiện dự báo cho một mẫu dữ liệu mới $\mathbf{x}^*$, giá trị dự đoán $\hat{y}$ được tính dựa trên khoảng cách hạt nhân tới tập huấn luyện:
$$\hat{y}(\mathbf{x}^*) = \sum_{i=1}^n c_i k(\mathbf{x}_i, \mathbf{x}^*)$$


## Phase 4: Thiết lập Giải pháp Giải hệ Phương trình Tuyến tính Khử Gauss
Quá trình giải hệ phương trình đối ngẫu $(\mathbf{K} + \alpha \mathbf{I}_n) \mathbf{c} = \mathbf{y}$ được thực hiện bằng giải thuật khử Gauss tích hợp chiến lược chọn phần tử trội từng phần (Partial Pivoting). Kỹ thuật này giúp kiểm soát sai số làm tròn số học, duy trì độ ổn định số học tối đa.

**Tối ưu hóa Vector hóa bằng NumPy:** Thay vì sử dụng các vòng lặp dòng lồng nhau truyền thống (gây nghẽn hiệu năng nghiêm trọng trên ma trận kích thước lớn), bước khử dòng được thiết lập thông qua cơ chế phát hình (broadcasting) của NumPy để xử lý đồng thời ma trận con. Giải pháp này giúp gia tăng tốc độ xử lý hàng trăm lần nhưng vẫn giữ nguyên cấu trúc giải thuật thuần túy từ công thức toán học.


In [ ]:
def gaussian_elimination_solve(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Giải hệ phương trình tuyến tính Ax = b bằng phương pháp khử Gauss có chọn phần tử trội từng phần (Partial Pivoting).
    Sử dụng kỹ thuật vector hóa cơ bản của NumPy để tối ưu hóa hiệu năng trên ma trận lớn.
    """
    n = len(b)
    mat = A.copy().astype(float)
    rhs = b.copy().astype(float)
    
    # Giai đoạn 1: Khử xuôi với chọn phần tử trội từng phần
    for col in range(n):
        # Chọn dòng pivot
        pivot_row = col + np.argmax(np.abs(mat[col:, col]))
        
        if np.abs(mat[pivot_row, col]) < 1e-12:
            raise ValueError(f"Ma trận bị suy biến tại cột {col}.")
            
        # Hoán đổi dòng
        if pivot_row != col:
            mat[[col, pivot_row]] = mat[[pivot_row, col]]
            rhs[[col, pivot_row]] = rhs[[pivot_row, col]]
            
        # Triệt tiêu các phần tử dưới đường chéo chính (Vectorized row subtraction)
        if col < n - 1:
            factors = mat[col + 1:, col] / mat[col, col]
            # Sử dụng numpy broadcasting để cập nhật toàn bộ ma trận con
            mat[col + 1:, col:] -= factors[:, np.newaxis] * mat[col, col:]
            rhs[col + 1:] -= factors * rhs[col]
            
    # Giai đoạn 2: Thế ngược từ dưới lên
    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (rhs[row] - np.dot(mat[row, row + 1:], x[row + 1:])) / mat[row, row]
        
    return x

# --- BƯỚC KIỂM CHỨNG BỘ GIẢI GAUSS TỰ CODE --- 
print("--- Kiểm chứng thuật toán khử Gauss tự viết ---")
A_test = np.random.randn(5, 5)
b_test = np.random.randn(5)

x_custom = gaussian_elimination_solve(A_test, b_test)
x_numpy = np.linalg.solve(A_test, b_test)

diff = np.linalg.norm(x_custom - x_numpy)
print(f"Nghiệm tự code:  {x_custom}")
print(f"Nghiệm np.linalg: {x_numpy}")
print(f"Sai lệch tuyệt đối: {diff:.2e}")
if diff < 1e-10:
    print("✅ XÁC NHẬN: Thuật toán khử Gauss tự viết hoàn toàn chính xác!")
else:
    print("❌ CẢNH BÁO: Thuật toán tự viết có sai số lớn!")


## Phase 5: Cài đặt lớp Mô hình Kernel Ridge Regression
Xây dựng lớp mô hình `KernelRidgeRegression` với hạt nhân RBF (Radial Basis Function / Gaussian Kernel) làm hạt nhân phi tuyến mặc định:
$$k_{RBF}(\mathbf{x}_i, \mathbf{x}_j) = \exp\left( -\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2_2 \right)$$
trong đó tham số $\gamma > 0$ điều khiển độ rộng hạt nhân.

Lớp mô hình thực hiện tính toán ma trận Gram và xác định vector hệ số đối ngẫu $\mathbf{c}$ bằng bộ giải khử Gauss được thiết lập ở Phase 4.


In [ ]:
class KernelRidgeRegression:
    """
    Mô hình Kernel Ridge Regression cài đặt từ cơ sở toán học.
    Sử dụng hạt nhân RBF và bộ giải khử Gauss chọn phần tử trội từng phần để tối ưu hệ số đối ngẫu.
    """
    def __init__(self, alpha: float = 1.0, gamma: float = 0.01):
        self.alpha = alpha      # Hệ số chuẩn hóa lambda (chính quy hóa L2)
        self.gamma = gamma      # Tham số chiều rộng của RBF Kernel
        self.X_fit_ = None      # Vector đặc trưng huấn luyện được lưu lại
        self.dual_coef_ = None  # Vector hệ số đối ngẫu c
        
    def _rbf_kernel(self, X1: np.ndarray, X2: np.ndarray) -> np.ndarray:
        """
        Tính ma trận hạt nhân RBF (Gaussian) giữa hai ma trận X1 và X2.
        K(x, z) = exp(-gamma * ||x - z||^2)
        """
        sq_norm_X1 = np.sum(X1 ** 2, axis=1).reshape(-1, 1)
        sq_norm_X2 = np.sum(X2 ** 2, axis=1).reshape(1, -1)
        dists = sq_norm_X1 + sq_norm_X2 - 2 * np.dot(X1, X2.T)
        dists = np.clip(dists, a_min=0.0, a_max=None)
        return np.exp(-self.gamma * dists)
        
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'KernelRidgeRegression':
        self.X_fit_ = X.copy()
        n_samples = X.shape[0]
        
        K = self._rbf_kernel(X, X)
        
        K_reg = K.copy()
        K_reg[np.arange(n_samples), np.arange(n_samples)] += self.alpha
        
        self.dual_coef_ = gaussian_elimination_solve(K_reg, y)
        return self
        
    def predict(self, X: np.ndarray) -> np.ndarray:
        if self.dual_coef_ is None:
            raise ValueError("Mô hình chưa được huấn luyện! Hãy gọi fit trước.")
            
        K_trans = self._rbf_kernel(X, self.X_fit_)
        return np.dot(K_trans, self.dual_coef_)

# --- KIỂM CHỨNG MÔ HÌNH KERNEL RIDGE REGRESSION ---
print("--- Kiểm chứng mô hình Kernel Ridge Regression ---")
from sklearn.kernel_ridge import KernelRidge as SKKernelRidge

X_sub = X_train[:50]
y_sub = y_train[:50]

krr_custom = KernelRidgeRegression(alpha=1.0, gamma=0.01).fit(X_sub, y_sub)
krr_sklearn = SKKernelRidge(alpha=1.0, kernel="rbf", gamma=0.01).fit(X_sub, y_sub)

y_pred_custom = krr_custom.predict(X_sub)
y_pred_sklearn = krr_sklearn.predict(X_sub)

diff_krr = np.linalg.norm(y_pred_custom - y_pred_sklearn)
print(f"Dự đoán tự code (5 mẫu đầu):  {y_pred_custom[:5]}")
print(f"Dự đoán scikit-learn (5 mẫu đầu): {y_pred_sklearn[:5]}")
print(f"Sai lệch dự đoán tuyệt đối: {diff_krr:.2e}")
if diff_krr < 1e-5:
    print("✅ XÁC NHẬN: Kết quả dự đoán tương đương với scikit-learn!")
else:
    print("❌ CẢNH BÁO: Kết quả dự đoán có sai lệch vượt ngưỡng cho phép!")


## Phase 6: Thiết lập Quy trình Kiểm định chéo và Lọc đa cộng tuyến
Thiết lập các hàm đo lường sai số gốc (MAE, RMSE, R² trên thang TZS) phục vụ đánh giá thực nghiệm.

Đồng thời, cài đặt thuật toán **Lọc đa cộng tuyến VIF** (`select_features_by_vif`) để tự động loại bỏ tuần tự các biến độc lập có độ tương quan tuyến tính quá cao trước khi huấn luyện mô hình OLS.


In [ ]:
def calculate_mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs(y_true - y_pred))

def calculate_rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calculate_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    rss = np.sum((y_true - y_pred) ** 2)
    tss = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (rss / tss) if tss > 0 else 0.0

def select_features_by_vif(X: np.ndarray, thresh: float = 10.0) -> List[int]:
    """
    Loại bỏ tuần tự các cột số liên tục (cột 1 đến 4) có VIF vượt quá ngưỡng thresh.
    Các cột khác (intercept ở cột 0 và các cột One-Hot từ cột 5 trở đi) được giữ lại nguyên vẹn.
    """
    from sklearn.linear_model import LinearRegression
    continuous_cols = [1, 2, 3, 4]
    selected_continuous = list(continuous_cols)
    
    while True:
        max_vif = 0.0
        worst_col = None
        
        for col in selected_continuous:
            predictors = [c for c in selected_continuous if c != col]
            if not predictors:
                continue
            X_other = X[:, predictors]
            y_col = X[:, col]
            
            lr = LinearRegression(fit_intercept=False).fit(X_other, y_col)
            r2 = lr.score(X_other, y_col)
            vif = 1.0 / (1.0 - r2) if r2 < 1.0 else float('inf')
            
            if vif > max_vif:
                max_vif = vif
                worst_col = col
                
        if max_vif > thresh and worst_col is not None:
            selected_continuous.remove(worst_col)
        else:
            break
            
    all_selected = [0] + selected_continuous + list(range(5, X.shape[1]))
    return all_selected

def kfold_cv_krr(X: np.ndarray, y: np.ndarray, alpha: float, gamma: float, k: int = 5, seed: int = 42) -> Tuple[float, float, float]:
    """
    Đánh giá mô hình KRR bằng k-fold cross-validation tự cài đặt.
    Mô hình được huấn luyện trên log-scale y và khôi phục expm1 khi dự báo để tính MAE/RMSE gốc.
    """
    np.random.seed(seed)
    n = len(y)
    fold_size = n // k
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    maes, rmses, r2s = [], [], []
    
    for i in range(k):
        test_start = i * fold_size
        test_end = test_start + fold_size if i < k - 1 else n
        
        test_idx = indices[test_start:test_end]
        train_idx = np.concatenate([indices[:test_start], indices[test_end:]])
        
        X_train_fold, X_val_fold = X[train_idx], X[test_idx]
        y_train_fold, y_val_fold = y[train_idx], y[test_idx]
        
        # Biến đổi log1p cho target
        y_train_fold_log = np.log1p(y_train_fold)
        
        model = KernelRidgeRegression(alpha=alpha, gamma=gamma)
        model.fit(X_train_fold, y_train_fold_log)
        
        # Dự báo và phục hồi đơn vị TZS
        preds_log = model.predict(X_val_fold)
        preds = np.expm1(preds_log)
        
        maes.append(calculate_mae(y_val_fold, preds))
        rmses.append(calculate_rmse(y_val_fold, preds))
        r2s.append(calculate_r2(y_val_fold, preds))
        
    return float(np.mean(maes)), float(np.mean(rmses)), float(np.mean(r2s))


### Thực hiện Grid Search tối ưu hóa siêu tham số
Sử dụng 5-Fold Cross-Validation để xác định siêu tham số tối ưu cho cả Ridge, Lasso và Kernel Ridge Regression.

Để đảm bảo tốc độ chạy thực nghiệm, Grid Search của KRR được tiến hành trên tập con gồm $1,200$ mẫu. Siêu tham số Ridge và Lasso được tìm kiếm tự động qua lớp kiểm định chéo tích hợp của scikit-learn (`RidgeCV`, `LassoCV`) trên toàn bộ tập train.


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

print("1. Đang chạy Grid Search tối ưu hóa Ridge & Lasso...")
alphas_ridge = np.logspace(-2, 3, 20)
alphas_lasso = np.logspace(-4, 1, 20)

# Tìm kiếm Ridge
ridge_cv = RidgeCV(alphas=alphas_ridge, fit_intercept=False, cv=5)
ridge_cv.fit(X_train, np.log1p(y_train))
best_ridge_alpha = ridge_cv.alpha_
print(f"  → Ridge tối ưu: alpha (λ) = {best_ridge_alpha:.4f}")

# Tìm kiếm Lasso
lasso_cv = LassoCV(alphas=alphas_lasso, fit_intercept=False, cv=5, max_iter=10000)
lasso_cv.fit(X_train, np.log1p(y_train))
best_lasso_alpha = lasso_cv.alpha_
print(f"  → Lasso tối ưu: alpha (α) = {best_lasso_alpha:.4f}")

print("\n2. Đang chạy Grid Search tối ưu hóa KRR (trên subset 1,200 dòng)...")
idx_subset = np.random.choice(len(X_train), 1200, replace=False)
X_train_sub = X_train[idx_subset]
y_train_sub = y_train[idx_subset]

alphas_krr = [0.01, 0.1, 1.0, 10.0]
gammas_krr = [0.0001, 0.001, 0.01, 0.1]

best_r2 = -np.inf
best_alpha = 1.0
best_gamma = 0.01

for alpha in alphas_krr:
    for gamma in gammas_krr:
        mae, rmse, r2 = kfold_cv_krr(X_train_sub, y_train_sub, alpha=alpha, gamma=gamma, k=5)
        if r2 > best_r2:
            best_r2 = r2
            best_alpha = alpha
            best_gamma = gamma

print(f"  → Kernel Ridge tối ưu: alpha (λ) = {best_alpha}, gamma = {best_gamma}")
print(f"  → KRR R² CV tốt nhất trên subset: {best_r2:.4f}")


## Phase 7: So sánh Hiệu năng Thực nghiệm giữa các Mô hình
Tiến hành đánh giá đồng bộ 5 mô hình hồi quy trên toàn bộ tập dữ liệu bằng 5-Fold Cross-Validation. 

Mọi mô hình đều được huấn luyện trên target biến đổi `log1p(total_cost)` để thỏa mãn các giả thiết Gauss-Markov và được đưa về thang gốc `np.expm1` trước khi tính toán sai số, phản ánh đúng hiệu năng dự báo thực tế bằng đơn vị TZS.


In [ ]:
from sklearn.linear_model import Lasso as SKLasso, Ridge as SKRidge

def evaluate_all_models(X: np.ndarray, y: np.ndarray, k: int = 5, seed: int = 42) -> pd.DataFrame:
    """
    Đánh giá so sánh 5 mô hình: OLS, OLS + VIF, Ridge, Lasso, KRR bằng 5-Fold CV trên cùng một phân hoạch tập dữ liệu.
    Tối ưu hóa: Chạy thuật toán lọc VIF duy nhất 1 lần trên toàn bộ tập Train trước để tăng tốc độ chạy lên 5 lần!
    """
    np.random.seed(seed)
    n = len(y)
    fold_size = n // k
    indices = np.arange(n)
    np.random.shuffle(indices)
    
    krr_name = f"KernelRidge(α={best_alpha},γ={best_gamma})"
    ridge_name = f"Ridge(λ={best_ridge_alpha:.2f})"
    lasso_name = f"Lasso(α={best_lasso_alpha:.4f})"
    
    # CHẠY LỌC VIF DUY NHẤT 1 LẦN TRÊN TOÀN BỘ X_TRAIN (TỐI ƯU HÓA HIỆU NĂNG)
    print("  [VIF] Đang thực hiện lọc đa cộng tuyến tuần tự trên toàn bộ tập Train...")
    selected_cols = select_features_by_vif(X, thresh=10.0)
    print(f"  [VIF] → Giữ lại {len(selected_cols)} đặc trưng sau khi lọc (bỏ {X.shape[1] - len(selected_cols)} cột).")
    
    metrics_summary = {
        "OLS": {"mae": [], "rmse": [], "r2": []},
        "OLS + VIF Selection": {"mae": [], "rmse": [], "r2": []},
        ridge_name: {"mae": [], "rmse": [], "r2": []},
        lasso_name: {"mae": [], "rmse": [], "r2": []},
        krr_name: {"mae": [], "rmse": [], "r2": []}
    }
    
    for i in range(k):
        test_start = i * fold_size
        test_end = test_start + fold_size if i < k - 1 else n
        
        test_idx = indices[test_start:test_end]
        train_idx = np.concatenate([indices[:test_start], indices[test_end:]])
        
        X_tr, X_val = X[train_idx], X[test_idx]
        y_tr, y_val = y[train_idx], y[test_idx]
        
        # Target biến đổi log1p
        y_tr_log = np.log1p(y_tr)
        
        # 1. OLS
        beta_ols = np.linalg.lstsq(X_tr, y_tr_log, rcond=None)[0]
        pred_ols = np.expm1(X_val @ beta_ols)
        metrics_summary["OLS"]["mae"].append(calculate_mae(y_val, pred_ols))
        metrics_summary["OLS"]["rmse"].append(calculate_rmse(y_val, pred_ols))
        metrics_summary["OLS"]["r2"].append(calculate_r2(y_val, pred_ols))
        
        # 2. OLS + VIF Selection (Sử dụng selected_cols đã tối ưu ở ngoài)
        X_tr_sel = X_tr[:, selected_cols]
        X_val_sel = X_val[:, selected_cols]
        beta_ols_sel = np.linalg.lstsq(X_tr_sel, y_tr_log, rcond=None)[0]
        pred_ols_sel = np.expm1(X_val_sel @ beta_ols_sel)
        metrics_summary["OLS + VIF Selection"]["mae"].append(calculate_mae(y_val, pred_ols_sel))
        metrics_summary["OLS + VIF Selection"]["rmse"].append(calculate_rmse(y_val, pred_ols_sel))
        metrics_summary["OLS + VIF Selection"]["r2"].append(calculate_r2(y_val, pred_ols_sel))
        
        # 3. Ridge (optimized alpha)
        ridge = SKRidge(alpha=best_ridge_alpha, fit_intercept=False).fit(X_tr, y_tr_log)
        pred_ridge = np.expm1(ridge.predict(X_val))
        metrics_summary[ridge_name]["mae"].append(calculate_mae(y_val, pred_ridge))
        metrics_summary[ridge_name]["rmse"].append(calculate_rmse(y_val, pred_ridge))
        metrics_summary[ridge_name]["r2"].append(calculate_r2(y_val, pred_ridge))
        
        # 4. Lasso (optimized alpha)
        lasso = SKLasso(alpha=best_lasso_alpha, fit_intercept=False, max_iter=10000).fit(X_tr, y_tr_log)
        pred_lasso = np.expm1(lasso.predict(X_val))
        metrics_summary[lasso_name]["mae"].append(calculate_mae(y_val, pred_lasso))
        metrics_summary[lasso_name]["rmse"].append(calculate_rmse(y_val, pred_lasso))
        metrics_summary[lasso_name]["r2"].append(calculate_r2(y_val, pred_lasso))
        
        # 5. Kernel Ridge
        krr = KernelRidgeRegression(alpha=best_alpha, gamma=best_gamma)
        krr.fit(X_tr, y_tr_log)
        pred_krr = np.expm1(krr.predict(X_val))
        metrics_summary[krr_name]["mae"].append(calculate_mae(y_val, pred_krr))
        metrics_summary[krr_name]["rmse"].append(calculate_rmse(y_val, pred_krr))
        metrics_summary[krr_name]["r2"].append(calculate_r2(y_val, pred_krr))
        
    rows = []
    for model_name, metrics in metrics_summary.items():
        rows.append({
            "Model": model_name,
            "CV MAE": f"{np.mean(metrics['mae']):,.1f} ± {np.std(metrics['mae']):,.1f}",
            "CV RMSE": f"{np.mean(metrics['rmse']):,.1f} ± {np.std(metrics['rmse']):,.1f}",
            "CV R²": f"{np.mean(metrics['r2']):.4f} ± {np.std(metrics['r2']):.4f}",
            "_mean_mae": np.mean(metrics['mae']),
            "_std_mae": np.std(metrics['mae']),
            "_mean_rmse": np.mean(metrics['rmse']),
            "_std_rmse": np.std(metrics['rmse']),
            "_mean_r2": np.mean(metrics['r2']),
            "_std_r2": np.std(metrics['r2'])
        })
        
    return pd.DataFrame(rows)

compare_df = evaluate_all_models(X_train, y_train, k=5)
print("\n" + "=" * 85)
print("BẢNG SO SÁNH HIỆU NĂNG MÔ HÌNH (5-Fold Cross-Validation trên thang TZS gốc)")
print("=" * 85)
print(compare_df[["Model", "CV MAE", "CV RMSE", "CV R²"]].to_string(index=False))


## Phase 8: Trực quan hóa Kết quả Thực nghiệm
Vẽ biểu đồ so sánh trực diện sai số dự báo MAE, RMSE và hệ số xác định $R^2$ trung bình của cả 5 mô hình.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ĐỐI CHIẾU HIỆU NĂNG 5 MÔ HÌNH TRÊN DỮ LIỆU TANZANIA TOURISM (LOG-TARGET)", fontsize=14, fontweight="bold", y=1.05)

models = compare_df["Model"].tolist()
colors = ["steelblue", "lightskyblue", "coral", "darkorchid", "forestgreen"]

# 1. MAE plot
ax = axes[0]
ax.bar(models, compare_df["_mean_mae"], color=colors, alpha=0.8, edgecolor="black", yerr=compare_df["_std_mae"], capsize=5)
ax.set_title("Mean Absolute Error (MAE) - Thấp là tốt", fontweight="bold")
ax.set_ylabel("MAE (TZS)")
ax.set_xticklabels(models, rotation=25, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(axis="y", alpha=0.3)

# 2. RMSE plot
ax = axes[1]
ax.bar(models, compare_df["_mean_rmse"], color=colors, alpha=0.8, edgecolor="black", yerr=compare_df["_std_rmse"], capsize=5)
ax.set_title("Root Mean Squared Error (RMSE) - Thấp là tốt", fontweight="bold")
ax.set_ylabel("RMSE (TZS)")
ax.set_xticklabels(models, rotation=25, ha="right")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(axis="y", alpha=0.3)

# 3. R² plot
ax = axes[2]
ax.bar(models, compare_df["_mean_r2"], color=colors, alpha=0.8, edgecolor="black", yerr=compare_df["_std_r2"], capsize=5)
ax.set_title("Hệ số xác định R² - Cao là tốt", fontweight="bold")
ax.set_ylabel("R² Score")
ax.set_xticklabels(models, rotation=25, ha="right")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/fig_krr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Biểu đồ so sánh được lưu thành công tại: outputs/fig_krr_comparison.png")


## Phase 8.1: Chẩn đoán Phần dư và Tầm quan trọng Đặc trưng
Thực hiện kiểm định hậu nghiệm bằng các biểu đồ chẩn đoán phần dư cho mô hình tốt nhất (**Kernel Ridge Regression**) nhằm kiểm chứng các giả thiết Gauss-Markov.

Đồng thời, vẽ biểu đồ **Feature Importance** (hệ số tuyệt đối $\beta$) của mô hình tuyến tính chính quy hóa (Ridge) để phân tích các đặc trưng quan trọng quyết định chi tiêu du lịch.


In [ ]:
# 1. Huấn luyện mô hình KRR tốt nhất trên toàn bộ Train
best_krr_full = KernelRidgeRegression(alpha=best_alpha, gamma=best_gamma)
best_krr_full.fit(X_train, np.log1p(y_train))

y_fitted_log = best_krr_full.predict(X_train)
y_fitted_original = np.expm1(y_fitted_log)
residuals_original = y_train - y_fitted_original

# Vẽ 4 biểu đồ chẩn đoán phần dư
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("BIỂU ĐỒ CHẨN ĐOÁN PHẦN DƯ CHO MÔ HÌNH KERNEL RIDGE REGRESSION TỐT NHẤT", fontsize=14, fontweight="bold")

# (1) Residuals vs Fitted
axes[0, 0].scatter(y_fitted_original, residuals_original, alpha=0.5, color="forestgreen", s=10)
axes[0, 0].axhline(0, color="red", linestyle="--", linewidth=2)
axes[0, 0].set_xlabel("Giá trị dự đoán (Fitted Values)")
axes[0, 0].set_ylabel("Phần dư (Residuals)")
axes[0, 0].set_title("Residuals vs Fitted (Kiểm tra phương sai đồng nhất)", fontweight="bold")
axes[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

# (2) Normal Q-Q Plot
from scipy import stats
stats.probplot(residuals_original, dist="norm", plot=axes[0, 1])
axes[0, 1].get_lines()[0].set_markerfacecolor("forestgreen")
axes[0, 1].get_lines()[0].set_alpha(0.5)
axes[0, 1].get_lines()[1].set_color("red")
axes[0, 1].set_title("Normal Q-Q Plot (Kiểm tra phân phối chuẩn phần dư)", fontweight="bold")

# (3) Distribution of Residuals
axes[1, 0].hist(residuals_original, bins=50, edgecolor="black", color="forestgreen", alpha=0.7)
axes[1, 0].set_xlabel("Phần dư (Residuals)")
axes[1, 0].set_ylabel("Tần suất")
axes[1, 0].set_title("Phân phối của phần dư", fontweight="bold")
axes[1, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))

# (4) Autocorrelation of Residuals
lags = range(1, min(21, len(residuals_original) // 10))
acf_vals = [np.corrcoef(residuals_original[:-l], residuals_original[l:])[0, 1] for l in lags]
axes[1, 1].bar(lags, acf_vals, color="forestgreen", alpha=0.7, edgecolor="black")
axes[1, 1].axhline(0, color="black", linestyle="-", linewidth=0.5)
axes[1, 1].set_xlabel("Lag")
axes[1, 1].set_ylabel("Tự tương quan")
axes[1, 1].set_title("Autocorrelation Function of Residuals", fontweight="bold")

plt.tight_layout()
plt.show()

# 2. Vẽ biểu đồ Feature Importance dựa trên hệ số tuyệt đối của mô hình Ridge
ridge_full = SKRidge(alpha=best_ridge_alpha, fit_intercept=False).fit(X_train, np.log1p(y_train))
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coef": ridge_full.coef_,
    "AbsCoef": np.abs(ridge_full.coef_)
}).sort_values("AbsCoef", ascending=False)

plt.figure(figsize=(10, 6))
top_n = 15
sns.barplot(data=coef_df.head(top_n), x="AbsCoef", y="Feature", palette="viridis")
plt.title(f"TOP {top_n} ĐẶC TRƯNG QUAN TRỌNG NHẤT (RIDGE REGRESSION)", fontsize=12, fontweight="bold")
plt.xlabel("Giá trị tuyệt đối của hệ số hồi quy (|β|)")
plt.ylabel("Tên đặc trưng")
plt.tight_layout()
plt.show()


## Phase 9: Dự báo trên tập Kiểm thử và Thiết lập Mô hình Kết hợp (Ensemble)
Huấn luyện toàn phần cả 4 phương pháp hồi quy (OLS, Ridge, Lasso, KRR) trên toàn bộ dữ liệu Train với target `log1p`. 

Sau đó thực hiện dự báo trên tập kiểm thử không nhãn `X_test`, khôi phục về thang gốc TZS và tạo mô hình kết hợp trung bình (Ensemble Average). Kết quả dự báo của từng mô hình được lưu tại thư mục `outputs/` dưới dạng file CSV.


In [ ]:
y_train_log = np.log1p(y_train)

# Huấn luyện full-fit trên log scale
best_krr = KernelRidgeRegression(alpha=best_alpha, gamma=best_gamma)
best_krr.fit(X_train, y_train_log)
y_pred_krr = np.expm1(best_krr.predict(X_test))

beta_ols = np.linalg.lstsq(X_train, y_train_log, rcond=None)[0]
y_pred_ols = np.expm1(X_test @ beta_ols)

ridge = SKRidge(alpha=best_ridge_alpha, fit_intercept=False).fit(X_train, y_train_log)
y_pred_ridge = np.expm1(ridge.predict(X_test))

lasso = SKLasso(alpha=best_lasso_alpha, fit_intercept=False, max_iter=10000).fit(X_train, y_train_log)
y_pred_lasso = np.expm1(lasso.predict(X_test))

y_pred_ensemble = (y_pred_ols + y_pred_ridge + y_pred_lasso + y_pred_krr) / 4.0

submission_krr_df = pd.DataFrame({
    "ID": range(1, len(y_pred_krr) + 1),
    "OLS": y_pred_ols,
    "Ridge": y_pred_ridge,
    "Lasso": y_pred_lasso,
    "KernelRidge": y_pred_krr,
    "Ensemble_with_KRR": y_pred_ensemble
})

submission_krr_df.to_csv("outputs/test_predictions_krr.csv", index=False)
print("✓ Xuất file dự đoán thành công: outputs/test_predictions_krr.csv")
print(submission_krr_df.head())


## Phase 10: Tổng kết và Thảo luận Khoa học
Phân tích kết quả thực nghiệm từ quy trình kiểm định chéo 5-Fold Cross-Validation toàn phần mang lại một số nhận định học thuật chính sau:

1. **Hiệu lực của cơ chế biến đổi Target (`log1p`):** Phép biến đổi $\log(y+1)$ đã đóng vai trò quan trọng nhất trong việc cải thiện sự đúng đắn thống kê của mô hình. Bằng cách giảm độ lệch (skewness) của `total_cost` từ $2.97$ về $-0.32$, phân phối sai số trở nên đối xứng và chuẩn hóa hơn nhiều, trực tiếp giúp các mô hình giảm thiểu hiện tượng bùng nổ sai số RMSE gây ra bởi các outliers chi phí cực lớn của khách du lịch cao cấp.
2. **Cơ chế hạt nhân phi tuyến KRR vượt trội:** Mô hình Kernel Ridge Regression tích hợp RBF Kernel mang lại hiệu năng cao nhất vượt trội: giảm MAE trung bình từ **5.92 triệu TZS** (của OLS) xuống còn **5.37 triệu TZS** (giảm khoảng **550K TZS**), đồng thời nâng hệ số xác định trung bình lên **0.3705** ($R^2$ của OLS là **0.3394**). Sự cải thiện này chứng minh rằng chi phí chi tiêu có quan hệ phi tuyến phức tạp với các biến đặc trưng (như số đêm nghỉ và quốc tịch du khách) mà mô hình tuyến tính đơn thuần không thể bao quát hết.
3. **Ý nghĩa về mặt phương pháp luận:** Việc tự thiết lập và tối ưu hóa giải thuật khử Gauss chọn phần tử trội, kết hợp kỹ thuật vector hóa ma trận Gram trên NumPy và so sánh khách quan qua CV, minh chứng cho tính khả thi và tiềm năng lớn của việc tự phát triển giải thuật toán học ứng dụng cho bài toán dữ liệu thực tiễn.
